# Convert Input vectors of size 3x32x32 to 64 tokens

In [25]:
import os
import sys
from pathlib import Path
import re

import torch
import zarr
import numpy as np
import pytorch_lightning as pl

from torch.utils.data import Dataset, DataLoader
from omegaconf import OmegaConf

# -------------------------
# Add external repos
# -------------------------
BASE = Path("/projappl/project_2012747/mars_derrick_branch/third_party")
CKPT_DIR = Path(
    "/scratch/project_2012747/Mars_Derrick/checkpoints/checkpoint_downsample_100"
)
ZIP_PATH = Path("/scratch/project_2012747/mars_data/order_batch_model/train/intermediate/")
OUT_DIR = Path("/scratch/project_2012747/mars_data/order_batch_model/train/final/")

sys.path.insert(0, str(BASE / "latent_diffusion"))
sys.path.insert(0, str(BASE / "taming-transformers"))


# -------------------------
# Lightning wrapper
# -------------------------
class VQForOrders(pl.LightningModule):
    def __init__(self, vqmodel):
        super().__init__()
        self.m = vqmodel

    def forward(self, x):
        q, _, _ = self.m.encode(x)
        return self.m.decode(q)


# -------------------------
# Load best checkpoint
# -------------------------
def find_best_checkpoint(ckpt_dir: Path) -> Path:
    ckpts = list(ckpt_dir.glob("*.ckpt"))

    def extract_val_loss(p):
        m = re.search(r"val_loss=([0-9]+(?:\.[0-9]+)?)", p.name)
        return float(m.group(1))

    return min(ckpts, key=extract_val_loss)


def load_model(best_ckpt):

    from ldm.util import instantiate_from_config

    cfg = OmegaConf.load(
        "/projappl/project_2012747/mars_derrick_branch/third_party/latent_diffusion/models/first_stage_models/vq-f4/config.yaml"
    )

    vq = instantiate_from_config(cfg.model)

    model = VQForOrders.load_from_checkpoint(
        checkpoint_path=best_ckpt,
        vqmodel=vq,
        strict=True
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.to(device)
    model.eval()

    return model, device


# -------------------------
# Fast Zarr dataset
# -------------------------
class ZarrDataset(Dataset):
    def __init__(self, arr):
        self.arr = arr

    def __len__(self):
        return self.arr.shape[0]

    def __getitem__(self, idx):
        x = self.arr[idx]

        # normalize on CPU (faster)
        return torch.from_numpy(x).float().div_(255.0)


# -------------------------
# CONFIG
# -------------------------
BATCH_SIZE = 512
NUM_WORKERS = 4   # adjust 4–16 on Puhti

OUT_DIR.mkdir(exist_ok=True)




# ---------- UNZIP ----------
import zipfile, shutil
from pathlib import Path

EXTRACT_DIR = ZIP_PATH.with_suffix("")

if not EXTRACT_DIR.exists():
    print("Unzipping...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR.parent)

# (your encoding code here)


# -------------------------
# Load model
# -------------------------
best_ckpt = find_best_checkpoint(CKPT_DIR)
model, device = load_model(best_ckpt)


# -------------------------
# Load Zarr (DIRECTORY, not ZIP)
# -------------------------
print("Opening Zarr...")

root = zarr.open(EXTRACT_DIR, mode="r")
arr = root["images"]

dataset = ZarrDataset(arr)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

num_samples = len(dataset)


# -------------------------
# Output Zarr (FAST)
# -------------------------
out_file = OUT_DIR / "tokens.zarr"

store = zarr.DirectoryStore(out_file)

tokens_arr = zarr.zeros(
    shape=(num_samples, 64),
    chunks=(BATCH_SIZE, 64),
    dtype="i4",
    store=store,
    overwrite=True
)

print("Starting encoding...")


num_samples = arr.shape[0]

for start in range(0, num_samples, BATCH_SIZE):

    end = min(start + BATCH_SIZE, num_samples)

    batch = arr[start:end]   # <-- FAST contiguous read

    x = torch.from_numpy(batch).float().div_(255.0).to(device, non_blocking=True)

    with torch.inference_mode(), torch.autocast("cuda"):
        _, _, info = model.m.encode(x)

    B = x.size(0)
    tokens = info[2].view(B, -1).cpu().numpy()

    tokens_arr[start:end] = tokens

    if start % (BATCH_SIZE * 50) == 0:
        print(f"{start}/{num_samples}")


print("Done.")
print("Saved:", out_file)
# ---------- DELETE ----------
shutil.rmtree(EXTRACT_DIR)



making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 64, 64) = 12288 dimensions.
making attention of type 'vanilla' with 512 in_channels
loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
VQLPIPSWithDiscriminator running with hinge loss.
Opening Zarr...
Starting encoding...
0/1358265
25600/1358265
51200/1358265
76800/1358265
102400/1358265
128000/1358265
153600/1358265
179200/1358265
204800/1358265
230400/1358265
256000/1358265
281600/1358265
307200/1358265
332800/1358265
358400/1358265
384000/1358265
409600/1358265
435200/1358265
460800/1358265
486400/1358265
512000/1358265
537600/1358265
563200/1358265
588800/1358265
614400/1358265
640000/1358265
665600/1358265
691200/1358265
716800/1358265
742400/1358265
768000/1358265
793600/1358265
819200/1358265
844800/1358265
870400/1358265
896000/1358265
921600/1358265
947200/1358265
972800/1358265
998400/1358265
1024000/1358265
1049600/1358265


KeyboardInterrupt: 

In [3]:
# process to parquet 
import os
import sys
from pathlib import Path
import re
import zarr
import zipfile
import shutil
import torch
import numpy as np
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from omegaconf import OmegaConf
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# -------------------------
# Add external repos
# -------------------------
BASE = Path("/projappl/project_2012747/mars_derrick_branch/third_party")
CKPT_DIR = Path("/scratch/project_2012747/Mars_Derrick/checkpoints/checkpoint_downsample_100")
ZIP_DIR = Path("/scratch/project_2012747/mars_data/order_batch_model/train/intermediate/")
OUT_DIR = Path("/scratch/project_2012747/mars_data/order_batch_model/train/final/")

sys.path.insert(0, str(BASE / "latent_diffusion"))
sys.path.insert(0, str(BASE / "taming-transformers"))

OUT_DIR.mkdir(exist_ok=True)

# -------------------------
# Lightning wrapper
# -------------------------
class VQForOrders(pl.LightningModule):
    def __init__(self, vqmodel):
        super().__init__()
        self.m = vqmodel

    def forward(self, x):
        q, _, _ = self.m.encode(x)
        return self.m.decode(q)

# -------------------------
# Load best checkpoint
# -------------------------
def find_best_checkpoint(ckpt_dir: Path) -> Path:
    ckpts = list(ckpt_dir.glob("*.ckpt"))
    def extract_val_loss(p):
        m = re.search(r"val_loss=([0-9]+(?:\.[0-9]+)?)", p.name)
        return float(m.group(1))
    return min(ckpts, key=extract_val_loss)

def load_model(best_ckpt):
    from ldm.util import instantiate_from_config
    cfg = OmegaConf.load(BASE / "latent_diffusion/models/first_stage_models/vq-f4/config.yaml")
    vq = instantiate_from_config(cfg.model)
    model = VQForOrders.load_from_checkpoint(
        checkpoint_path=best_ckpt,
        vqmodel=vq,
        strict=True
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()
    return model, device

# -------------------------
# Fast Zarr dataset
# -------------------------
class ZarrDataset(Dataset):
    def __init__(self, arr):
        self.arr = arr

    def __len__(self):
        return self.arr.shape[0]

    def __getitem__(self, idx):
        x = self.arr[idx]
        return torch.from_numpy(x).float().div_(255.0)

# -------------------------
# Processing function
# -------------------------
def process_zip_file(zip_path: Path, model, device, batch_size=512):
    print(f"Processing {zip_path.name} ...")

    extract_dir = zip_path.with_suffix("")
    if extract_dir.exists():
        shutil.rmtree(extract_dir)

    # unzip
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(zip_path.parent)
    
    # find the extracted folder
    extracted_items = list(zip_path.parent.glob(zip_path.stem.replace(".zip","")+"*"))
    if not extracted_items:
        # fallback: maybe zip contains a single folder, pick it
        extracted_items = [p for p in zip_path.parent.iterdir() if p.is_dir()]
    
    if not extracted_items:
        raise FileNotFoundError(f"No extracted folder found for {zip_path}")
    
    # pick the first folder (usually the .zarr folder)
    zarr_folder = extracted_items[0]
    print(f"Using extracted folder: {zarr_folder}")
    
    # open Zarr
    root = zarr.open(zarr_folder, mode="r")
    arr = root["images"]

    num_samples = arr.shape[0]

    all_tokens = []

    # dataloader
    dataset = ZarrDataset(arr)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

    with torch.inference_mode():
        for i, x in enumerate(loader):
            x = x.to(device, non_blocking=True)
            with torch.autocast("cuda"):
                _, _, info = model.m.encode(x)
            B = x.size(0)
            tokens = info[2].view(B, -1).cpu().numpy()
            all_tokens.append(tokens)

            if i % 20 == 0:
                print(f"{zip_path.name}: processed {min((i+1)*batch_size, num_samples)}/{num_samples}")

    # combine all batches
    all_tokens = np.vstack(all_tokens)

    # save as parquet
    out_file = OUT_DIR / f"{zip_path.stem.replace('_order_images','')}_64_vector.parquet"
    table = pa.Table.from_arrays([all_tokens[:, i] for i in range(all_tokens.shape[1])],
                                 names=[f"v{i}" for i in range(all_tokens.shape[1])])
    pq.write_table(table, out_file)
    print(f"Saved {out_file} ({all_tokens.shape[0]} rows, {all_tokens.shape[1]} cols)")

    # cleanup
    shutil.rmtree(extract_dir)

# -------------------------
# Main
# -------------------------
def main():
    best_ckpt = find_best_checkpoint(CKPT_DIR)
    model, device = load_model(best_ckpt)

    zip_files = list(ZIP_DIR.glob("*.zarr.zip"))
    print(f"Found {len(zip_files)} zip files to process.")

    for zip_file in zip_files:
        process_zip_file(zip_file, model, device)

if __name__ == "__main__":
    main()


making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 64, 64) = 12288 dimensions.
making attention of type 'vanilla' with 512 in_channels
loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
VQLPIPSWithDiscriminator running with hinge loss.
Found 270 zip files to process.
Processing GOOG_2025-11-04_order_images.zarr.zip ...
Using extracted folder: /scratch/project_2012747/mars_data/order_batch_model/train/intermediate/GOOG_2025-11-04_order_images.zarr.zip
GOOG_2025-11-04_order_images.zarr.zip: processed 512/6991649
GOOG_2025-11-04_order_images.zarr.zip: processed 10752/6991649
GOOG_2025-11-04_order_images.zarr.zip: processed 20992/6991649
GOOG_2025-11-04_order_images.zarr.zip: processed 31232/6991649
GOOG_2025-11-04_order_images.zarr.zip: processed 41472/6991649
GOOG_2025-11-04_order_images.zarr.zip: processed 51712/6991649
GOOG_2025-11-04_order_images.zarr.zip: processed 61952/6991649


KeyboardInterrupt: 

In [22]:
!nvidia-smi


Wed Feb 18 22:08:26 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.288.01             Driver Version: 535.288.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla V100-SXM2-32GB           On  | 00000000:8A:00.0 Off |                    0 |
| N/A   27C    P0              56W / 300W |  19753MiB / 32768MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

## Take a sample 64 vector and convert back to 3 dimensional (RGB) image of size 3x32x32

In [19]:
import torch
import zarr
from zarr.storage import ZipStore
from pathlib import Path

# --- path to your saved token Zarr file ---
zarr_file = OUT_DIR / "LOBSTER-GOOGL-2025-12-16_order_images.zarr_tokens.zarr.zip"

# --- load Zarr ---
store = ZipStore(str(zarr_file), mode="r")
tokens_arr = zarr.open(store, mode="r")

# --- pick a sample (e.g., first one) ---
tokens = torch.tensor(tokens_arr[0], dtype=torch.long)  # shape: (64,)

# --- reshape tokens back to grid (8x8) ---
tokens_grid = tokens.view(1, 8, 8)  # (1, 8, 8) batch dimension

# --- look up codebook embeddings ---
# model.m is your VQGAN
# codebook is usually nn.Embedding inside your VQGAN quantizer
codebook = model.m.quantize.embedding.weight  # shape: (8192, 3)
print(len(codebook))
# tokens → embeddings
z_q = codebook[tokens_grid]  # shape: (1, 8, 8, 3)

# --- permute to (B, C, H, W) for decoder ---
z_q = z_q.permute(0, 3, 1, 2).contiguous()  # (1, 3, 8, 8)

# --- decode to image ---
with torch.inference_mode():
    x_hat = model.m.decode(z_q)  # (1, 3, 32, 32), float tensor

# --- convert to numpy (optional) ---
x_hat_np = x_hat[0].cpu().numpy()
print(x_hat_np.shape)  # should be (3, 32, 32)


8192
(3, 32, 32)


/PUHTI_TYKKY_Quvj2Tb/miniforge/envs/env1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 64, 64) = 12288 dimensions.
making attention of type 'vanilla' with 512 in_channels


/users/edwardma/.local/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/users/edwardma/.local/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
VQLPIPSWithDiscriminator running with hinge loss.
Loaded /scratch/project_2012747/Data_zarr/Training_VQGAN/LOBSTER-TSLA-2025-12-22_order_images.zarr.zip with shape (5548388, 3, 32, 32)
Processed 50000/5548388
Processed 100000/5548388
Processed 150000/5548388
Processed 200000/5548388
Processed 250000/5548388
Processed 300000/5548388
Processed 350000/5548388
Processed 400000/5548388
Processed 450000/5548388
Processed 500000/5548388
Processed 550000/5548388
Processed 600000/5548388
Processed 650000/5548388
Processed 700000/5548388
Processed 750000/5548388
Processed 800000/5548388
Processed 850000/5548388
Processed 900000/5548388
Processed 950000/5548388
Processed 1000000/5548388
Processed 1050000/5548388
Processed 1100000/5548388
Processed 1150000/5548388
Processed 1200000/5548388
Processed 1250000/5548388
Processed 1300000/5548388
Processed 1350000/5548388
Processed 1400000/5548388
Processed 1450000/5548388
Proces